# Preprocessing in SynthEval

SynthEval is a final-stage evaluation tool, that is made to be applied to the final outputs of a synthesis pipeline / generative procedure. The intention is that data can be supplied in a variety of formats, and that SynthEval will take care of most of the necessary preprocessing to get the data into a format that can be evaluated.

In this codebook we will clarify what steps we take, and what steps we expect users to take, in order to get data into the right format for evaluation.

### What does vs. doesn't SynthEval do?

What we take care of:
- Inferring variable types; i.e. categorical, numerical if not specified by the user.
- Checking that there are no structural differences between the real and synthetic data (e.g. different variable names, different number of variables).
- Using the same preprocessing codes for the real training and testing data and for the synthetic data.
- Converting string variables to numeric representations (via label encoding).
- Converting numerical variables to categorical if they have a low number of unique values (e.g. 10 or less).
- Keeping labels and confounder variables separate when they are specified.

What we do not take care of:
- Date variables. If you have date variables in your data, you should convert them to a numeric representation before evaluation (e.g., by converting them to timestamps or by extracting features like year, month, day).
- Removing variables from the data. If you want to evaluate only a subset of the variables, you should subset the data before evaluation.
- Stratification of the data. If you want to evaluate on a specific subset of the data, you should subset the data before evaluation.
- Handling of missing values. If you have missing values in your data, you should handle them before evaluation. 

### Handling of missing values
While it would be great to be able to handle missingness in SynthEval more thoughtfully, the task has no one-size-fits-all solution in the field of synthetic data generation (e.g., do we want to evaluate how well the missingness is modelled, or do we want to evaluate the synthetic data only on the non-missing values). Therefore we settled for the safe solution that avoids unintentionally wrong analyses. In practice, SynthEval has an argument called `missing_directive` which will raise a `ValueError` on the default setting `'raise'`, if it detects that missing values are present in the data. A common case is that the training data contains some missing values, but that the synthetic data does not: If the user wishes to automatically drop rows in the training/test data with missing values, they can set `missing_directive = 'drop'`. Finally, if the users have taken their own steps to handle missing values, we allow users to bypass the check and proceed on their own risk, with the `missing_directive = 'ignore'`.

If you have generated synthetic data that mimics the missingness in the training data, most evlauation metrics should still work, but you will need to mask or replace the `None` values in the real and synthetic data (e.g., with a masking value like -9999) to avoid halting errors.

## What happens when I run an evaluation?
In the following we will go through the steps that happen when you run an evaluation in SynthEval, by showing some snippets from the codebase.

Let us assume you did:

In [1]:
import pandas as pd
from syntheval import SynthEval

df_real_train = pd.read_csv('example/penguins_train.csv')
df_real_test = pd.read_csv('example/penguins_test.csv')

df_syn = pd.read_csv('example/penguins_BN_syn.csv')

SE = SynthEval(df_real_train, holdout_dataframe=df_real_test)
res = SE.evaluate(df_syn, analysis_target = ['species'])

Rich console is not supported in this environment. Defaulting to ascii console.
Inferred categorical columns (unique threshold: 10):
['species', 'island', 'sex']
SynthEval: synthetic data read successfully


0it [00:00, ?it/s]

SynthEval results


What you will see first is the check that infers that some variables are categorical. This is done because we did not prespecify the categorical variables:
```python 
# SynthEval.__init__ lines: 146-152
if cat_cols is None:
    cat_cols = get_cat_variables(real_dataframe, unique_threshold)
    if self.verbose:
        print(f"Inferred categorical columns (unique threshold: {unique_threshold}):\n{cat_cols}")

self.categorical_columns = cat_cols
self.numerical_columns = [column for column in real_dataframe.columns if column not in cat_cols]
```
It is worth noting that if you are specifying the categorical variables yourself (as a list of column names), you need to specify *all* of the categorical variables, since this step is only done if `cat_cols` is `None`. If you specify some, but not all, of the categorical variables, the remaining ones will not be converted to categorical and may cause issues when they are otherwise treated as numerical variables in the evaluation.

The `get_cat_variables` function is a simple function that checks the variable from `pandas` DataFrame, and additionally enforces a threshold on the number of unique values in each numerical column, and returns the column names of those that have a number of unique values below a certain threshold (default is 10):

In [2]:
import numpy as np

# utils/variable_detection.py lines: 8-20
def get_cat_variables(df, threshold):
    cat_variables = []

    for col in df.columns:
        if df[col].dtype == "object":
            cat_variables.append(col)
        # https://stackoverflow.com/questions/37726830/how-to-determine-if-a-number-is-any-type-of-int-core-or-numpy-signed-or-not
        elif (
            np.issubdtype(df[col].dtype, np.integer) or np.issubdtype(df[col].dtype, np.floating)
        ) and df[col].nunique() < threshold:
            cat_variables.append(col)

    return cat_variables

# Example
get_cat_variables(df_real_train, threshold=10)

['species', 'island', 'sex']

The next step happens at the evaluation call once the synthetic data is supplied. Here we check that the synthetic data has the same columns as the real data, and enforce the same column ordering (in the case of a mismatch). Additionally we use the `consistent_label_encoding` module to ensure that the same categorical and numerical preprocessising is applied.

```python
# SynthEval.evaluate lines: 222-226
CLE = consistent_label_encoding(self.real, self.synt, self.categorical_columns, self.numerical_columns, self.hold_out)
real_data = CLE.encode(self.real)
synt_data = CLE.encode(self.synt)
if self.hold_out is not None: hout_data = CLE.encode(self.hold_out)
else: hout_data = None
```

The `consistent_label_encoding` module is a custom module stacks the real, synthetic, and optional holdout data together, and fits `sklearn`'s `OrdinalEncoder` and `MinMaxScaler` on the combined data, to ensure that every level of the categorical variables is encoded in the same way, and that all numerical variables are fit within the same normalised range.

In [7]:
from syntheval.utils.preprocessing import consistent_label_encoding
#Example output:
df_real = pd.DataFrame({
    'A': ['cat', 'dog', 'cat', 'mouse'],
    'B': [1, 2, 3, 4],
    'C': [1, 1, 1, 1]
})

df_syn = pd.DataFrame({
    'A': ['mouse', 'cat', 'dog'],
    'B': [3, 3, 2],
    'C': [1, 1, 1]
})

CLE = consistent_label_encoding(df_real, df_syn, categorical_columns=['A','C'], numerical_columns=['B'])

df_real_enc = CLE.encode(df_real)
print(df_real_enc)
df_syn_enc = CLE.encode(df_syn)
print(df_syn_enc)

   A         B  C
0  0  0.000000  0
1  1  0.333333  0
2  0  0.666667  0
3  2  1.000000  0
   A         B  C
0  2  0.666667  0
1  0  0.666667  0
2  1  0.333333  0


This encoding can be reversed if needed in the metric implementations since the CLE object is passed to the `MetricClass` object i.e.,

```python
# NovelMetricImplelentation.evaluate:
synt_data = CLE.decode(self.synt_data)
```